# CoReNet Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cms-cmu/CoReNet/blob/main/examples/quickstart.ipynb)

This notebook demonstrates how to load a configuration, initialize a **CoReNet** model, evaluate predictions, and train on the sample $HH \to b\bar{b}b\bar{b}$ dataset.

## 1. Setup & Imports

In [ ]:
from pathlib import Path

import corenet
from corenet import CoReNet, load_config, train
from corenet.data import H5Dataset

print(f"CoReNet version: {corenet.__version__}")

## 2. Load Configuration and Model Architecture

In [ ]:
config_path = Path("examples/hh4b/hh4b.yaml")
spec = load_config(config_path)
print(f"Classes: {spec.classes}")
print(f"Hypotheses: {list(spec.hypotheses.keys())}")

model = CoReNet(spec, device="cpu")
print(model)

## 3. Load Sample Dataset

In [ ]:
data_path = Path("examples/data/hh4b_small.h5")
dataset = H5Dataset(data_path, spec)
print(f"Loaded {len(dataset):,} events.")
for k, v in dataset.inputs.items():
    print(f"Input tensor {k:10s}: shape {tuple(v.shape)}")

## 4. Run Model Prediction (Inference)

In [ ]:
# Initialize input statistics
model.updateMeanStd(dataset.inputs)
model.initMeanStd()
model.eval()

# Run predict_proba for normalized softmax probabilities
probs = model.predict_proba(dataset.inputs)
print(f"Predicted probabilities shape: {probs.shape}")
print(f"Sample event predictions (first 5):\n{probs[:5]}")

## 5. Quick 1-Epoch Training Run

In [ ]:
out_dir = Path("runs/quickstart_demo")
out_dir.mkdir(parents=True, exist_ok=True)

ckpt = train.main(
    [
        "--config",
        str(config_path),
        "--input",
        str(data_path),
        "--fold",
        "0",
        "--exclude-folds",
        "1",
        "--output-dir",
        str(out_dir),
        "--epochs",
        "2",
        "--batch-size-init",
        "256",
        "--device",
        "cpu",
    ]
)
print(f"Model checkpoint saved to: {ckpt}")